# 行为树树状图（缩进示意）

> 本图是 `SentryBehaviorTree.xml` 的结构化缩进视图，用于快速对齐 Groot / BT.CPP 的节点层级。

## MainSentryTree（主树）

```
MainSentryTree
└─ Sequence
   ├─ SubTree: PerceptionAndBlackboard
   ├─ SubTree: InitOnce
   └─ WhileDoElse (IsMatchStage)
      ├─ THEN: ReactiveSequence
      │  ├─ SubTree: CommandHub
      │  └─ ReactiveFallback (priority)
      │     ├─ SubTree: DeathAndRespawn
      │     ├─ SubTree: WeaknessRecovery
      │     ├─ SubTree: CriticalSurvival
      │     ├─ SubTree: BaseDefense
      │     ├─ SubTree: EngageCombat
      │     ├─ SubTree: SustainAndEconomy
      │     ├─ SubTree: ObjectivePlanner
      │     └─ SubTree: PatrolAndScan
      └─ ELSE: ReactiveSequence
         ├─ RateController(1Hz) -> SendGoal(Home)
         ├─ RobotControl(fire off)
         └─ RateController(2Hz) -> SentryCmdMux(posture=Move)
```

**优先级说明**（ReactiveFallback 从上到下递减）：

| 优先级 | 子树 | 职责 |
|--------|------|------|
| 0 (最高) | DeathAndRespawn | 死亡/复活处理 |
| 1 | WeaknessRecovery | 虚弱解除 |
| 2 | CriticalSurvival | 危急生存（HP/热量） |
| 3 | BaseDefense | 基地防御 |
| 4 | EngageCombat | 交战 |
| 5 | SustainAndEconomy | 补血/补弹 |
| 6 | ObjectivePlanner | 目标控制 |
| 7 (最低) | PatrolAndScan | 巡逻扫描 |

## 感知 & 初始化子树

### PerceptionAndBlackboard

```
PerceptionAndBlackboard
└─ Sequence
   ├─ SubGameStatus
   ├─ SubRobotStatus
   ├─ SubRFIDStatus
   ├─ SubRobotPosition
   ├─ SubRadarTracks
   └─ ParseSentryBlackboard
```

### CommandHub

```
CommandHub
└─ Sequence
   ├─ DecidePosture -> {cmd.posture}
   ├─ DecideEconomyCmd -> {cmd.allow_ammo_target, cmd.trig_remote_*}
   ├─ DecideRespawnCmd -> {cmd.confirm_*}
   ├─ RateController(5Hz) -> SentryCmdMux(0x0120)
   └─ KeepRunning
```

## 生存 & 防御子树

### DeathAndRespawn

```
DeathAndRespawn
└─ Sequence
   ├─ IsDead
   ├─ CancelNavGoal
   ├─ RobotControl(fire off)
   └─ KeepRunning
```

### WeaknessRecovery

```
WeaknessRecovery
└─ ReactiveSequence
   ├─ IsWeakness
   ├─ SelectNearestDispelCard -> {nav.goal_x,y}
   ├─ RateController(1Hz) -> SendGoal(DispelWeakness)
   ├─ RobotControl(fire off)
   └─ ReactiveFallback
      ├─ IsAnyDispelCardDetected
      └─ MoveAround(micro adjust)
```

### CriticalSurvival

```
CriticalSurvival
└─ ReactiveSequence
   ├─ IsCriticalState(HP or Heat)
   ├─ CancelNavGoal
   ├─ RobotControl(fire off)
   ├─ SelectSafeRetreatGoal -> {nav.goal_x,y}
   ├─ RateController(1Hz) -> SendGoal(Retreat)
   └─ KeepRunning
```

### BaseDefense

```
BaseDefense
└─ ReactiveSequence
   ├─ IsBaseThreatened
   ├─ RobotControl(fire on)
   ├─ RateController(1Hz) -> SendGoal(DefendAnchor)
   └─ SubTree: CombatLoop
```

## 战斗子树

### EngageCombat

```
EngageCombat
└─ ReactiveSequence
   ├─ HasValidTarget
   ├─ IsCombatAllowed
   ├─ RobotControl(spin, fire on)
   └─ SubTree: CombatLoop
```

### CombatLoop

```
CombatLoop
└─ ReactiveSequence
   ├─ SelectBestTarget -> {combat.best_target}
   ├─ AimAtTarget
   ├─ ReactiveFallback
   │  ├─ IsFireWindowOk
   │  └─ RobotControl(fire off)
   ├─ FireBurst(burst/pause)
   └─ KeepRunning
```

## 后勤子树

### SustainAndEconomy

```
SustainAndEconomy
└─ ReactiveFallback
   ├─ SubTree: HealPlan
   └─ SubTree: AmmoPlan
```

### HealPlan

```
HealPlan
└─ ReactiveSequence
   ├─ IsHPBelow
   └─ ReactiveFallback
      ├─ (At SUPPLY) -> HoldAndHeal
      └─ GoSupply -> SendGoal + KeepRunning
```

### AmmoPlan

```
AmmoPlan
└─ ReactiveSequence
   ├─ IsAmmoBelow
   └─ ReactiveFallback
      ├─ (At SUPPLY) -> HoldForSupplyAmmoTick
      └─ SelectNearestResupplyStation -> SendGoal + KeepRunning
```

## 目标 & 巡逻子树

### ObjectivePlanner

```
ObjectivePlanner
└─ ReactiveSequence
   ├─ SelectObjective -> {nav.objective, nav.goal_x,y}
   ├─ RateController(1Hz) -> SendGoal(objective)
   ├─ RobotControl(fire off)
   ├─ (Arrive) IsAtGoal
   └─ HoldObjective(hold_ms)
```

### PatrolAndScan

```
PatrolAndScan
└─ Sequence
   ├─ RobotControl(scan, fire off)
   ├─ WaypointPatrol -> {nav.goal_x,y}
   ├─ RateController(1Hz) -> SendGoal(Patrol)
   └─ KeepRunning
```